<a href="https://colab.research.google.com/github/SampMark/Deep-Learning/blob/main/Training_Loops_with_KerasTuner_Consolided.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Loops de Treinamento Avançados e Ajuste de Hiperparâmetros com Keras e KerasTuner**


A evolução das técnicas de **loops de treinamento personalizados** e **otimização de hiperparâmetros** reflete a busca contínua por modelos de aprendizado de máquina mais eficientes, flexíveis e escaláveis. Algumas  abordagens transformaram o desenvolvimento de modelos, permitindo maior controle sobre o processo de treinamento e a exploração sistemática do espaço de hiperparâmetros. A seguir, buscamos aprofundar sua evolução, desafios e o equilíbrio entre complexidade computacional e desempenho, com base em algumas referências científicas e práticas atuais.

Vale salientar que o treinamento de modelos grandes consome energia equivalente a dezenas de carros (Schwartz et al., 2020), portanto, técnicas de otimização são cruciais para reduzir essa pegada de carbono.

### **1. Evolução dos Loops de Treinamento Personalizados**

#### **Contexto Histórico**
Antes da popularização de frameworks como **TensorFlow** e **PyTorch**, o desenvolvimento de modelos de *machine learning* era limitado por bibliotecas de alto nível que ocultavam detalhes do processo de treinamento. A introdução de ferramentas como **Keras** simplificou o treinamento com métodos como `.fit()`, mas também evidenciou a necessidade de flexibilidade para casos complexos, como redes adversárias generativas (GANs) e aprendizado por reforço.

#### **Avanços Técnicos**
- **AutoDiferenciação (AutoDiff)**
  A API `tf.GradientTape` no TensorFlow e o `torch.autograd` no PyTorch permitem capturar operações em tempo real para calcular gradientes, essencial para loops personalizados, substituindo métodos manuais de retropropagação, reduzindo erros e complexidade.
- **Integração com hardware robusto**
  Frameworks modernos otimizam loops personalizados para GPUs e TPUs, aproveitando paralelismo em larga escala. Por exemplo, o **JAX** combina diferenciação automática com compilação XLA para acelerar cálculos.
- **Flexibilidade em arquiteturas**
  Loops personalizados permitem implementar arquiteturas não convencionais, como redes neurais híbridas (ex.: combinando, por exemplo, CNNs e GNNs) ou algoritmos de otimização customizados (ex.: *lookahead optimizers*).

#### **Aplicações Avançadas**
- **Treinamento distribuído**
  Frameworks como **Horovod** e **TensorFlow Distributed** usam loops personalizados para sincronizar gradientes em múltiplas GPUs ou nós.
- **Aprendizado "federado"**
  O **TensorFlow Federated** implementa loops personalizados para treinar modelos em dados descentralizados, respeitando privacidade.

### **2. Otimização de Hiperparâmetros: da busca aleatória à AutoML**

#### **Evolução dos Métodos**
- **Busca aleatória e em Grade (_Grid Search_)**: foram métodos iniciais, porém ineficientes para espaços de hiperparâmetros grandes.
- **Otimização Bayesiana**: algoritmos como **Gaussian Processes** (Snoek et al., 2012) e **Tree-structured Parzen Estimators (TPE)** (Bergstra et al., 2011) priorizam regiões promissoras do espaço de busca.
- **Hyperband:** Proposto por Li et al. (2017), combina busca aleatória com early-stopping para acelerar a convergência.
- **Neural Architecture Search (NAS):** Ferramentas como **AutoKeras** e **Google AutoML** automatizam a escolha de arquiteturas e hiperparâmetros, usando técnicas como reforço (Zoph et al., 2018) e evolucionárias (Real et al., 2017).

#### **Ferramentas Modernas**
- **Keras Tuner:** Integra-se nativamente ao TensorFlow, suportando Random Search, Hyperband e Bayesian Optimization.
- **Optuna:** Framework flexível com suporte a otimização multi-objetivo e distribuída.
- **Ray Tune:** Escala buscas para clusters, combinando Hyperband com PBT (*Population-Based Training*).


### **3. Equilíbrio entre Complexidade Computacional e Desempenho**
#### **Desafios Centrais**
1. **Trade-off Precisão vs. Eficiência:**
   - Modelos maiores (ex.: BERT, ResNet-152) alcançam alta precisão, mas exigem recursos significativos. Técnicas como **podagem** e **quantização** (TensorFlow Model Optimization Toolkit) reduzem tamanho sem perda crítica de desempenho.
   - Exemplo: O modelo **MobileNetV2** usa convoluções profundas separáveis para equilibrar precisão e eficiência em dispositivos móveis.

2. **Otimização para Edge Computing:**
   - **Quantização Pós-Treinamento:** Converte pesos de `float32` para `int8`, reduzindo uso de memória em até 4x (TensorFlow Lite).
   - **Destilação de Conhecimento:** Treina modelos leves (*student models*) para imitar modelos grandes (*teacher models*), como no **DistilBERT**.

3. **Uso Eficiente de Recursos:**
   - **Treinamento com Precisão Mista:** Usa `float16`/`bfloat16` para reduzir uso de memória e acelerar cálculos em GPUs com Tensor Cores (ex.: NVIDIA RTX).
   - **Early-Stopping:** Interrompe treinamentos ruins cedo, economizando ciclos de CPU/GPU.



### **4. Impletação de Loops de Treinamento Personalizados**

Neste notebook será demonstrado como implementar loops de treinamento personalizados no Keras para maior controle, e será abordado como usar o KerasTuner para otimizar hiperparâmetros de modelos.
  * **Bibliotecas**: TensorFlow, Keras, KerasTuner, Scikit-learn, NumPy.
  
  * **Estrutura:**
    1.  **Parte 1**: implementando um Loop de Treinamento Personalizado para o MNIST.
    2.  **Parte 2**: ajuste de Hiperparâmetros com KerasTuner usando `model.fit()`.

O notebook visa demonstrar flexibilidade com loops de treinamento personalizados e o poder do KerasTuner para otimização automática, integrando os seguintes aspectos:
   * Controle fino vs. automação.
   * Importância da definição adequada das métricas.
   * Complexidade ao integrar KerasTuner com loops personalizados.

---

In [1]:
# Aloque uma GPU
!nvidia-smi

Thu May 15 19:12:55 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   51C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install keras_tuner

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.1/129.1 kB 8.4 MB/s eta 0:00:00


In [3]:
# --- Bibliotecas necessárias ---
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Flatten, Dense, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import SparseCategoricalCrossentropy
from tensorflow.keras.metrics import SparseCategoricalAccuracy, Mean
import keras_tuner as kt
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_classification

In [4]:
# --- Configurações Globais ---
RANDOM_SEED = 42
BATCH_SIZE = 64
NUM_EPOCHS = 10
LEARNING_RATE = 1e-3

# Semente para reprodutibilidade
tf.random.set_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

## **Parte 1: Loop de Treinamento Personalizado para MNIST**

### **Etapa 1: Carregamento do dataset MNIST e preparação dos Dados**

1. **Carregamento do Dataset MNIST:** Uso de `tf.keras.datasets.mnist.load_data()`.
2. **Pré-processamento:**
  * Normalização dos valores dos pixels (divisão por 255.0).
  * Divisão do conjunto de treino original em subconjuntos de treino e validação usando `train_test_split`.
  * Exibição das dimensões dos datasets para verificação.
3. **Criação de `tf.data.Dataset`:** Transformação dos arrays NumPy em objetos `tf.data.Dataset` para otimizar o pipeline de dados com:
  * `from_tensor_slices()`: Criação do dataset.
  * `shuffle()`: Embaralhamento dos dados de treino.
  * `batch()`: Agrupamento dos dados em lotes.
  * `prefetch()`: Pré-busca de lotes para otimizar o uso da CPU/GPU.

In [5]:
# Etapa 1. Carregamento do dataset MNIST e preparação dos Dados
(x_train_full, y_train_full), (x_test, y_test) = mnist.load_data()

# Normalização
x_train_full = x_train_full.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# Divisão do conjunto de treinamento completo em treino e validação
# O MNIST já vem com x_test, y_test, será criado um val_dataset do x_train_full
x_train, x_val, y_train, y_val = train_test_split(
    x_train_full, y_train_full, test_size=0.1, random_state=RANDOM_SEED
)
print("### Carregando e exibindo informações sobre o MNIST ###")
print(f"Dados de treino: {x_train.shape}, Rótulos de treino: {y_train.shape}")
print(f"Dados de validação: {x_val.shape}, Rótulos de validação: {y_val.shape}")
print(f"Dados de teste: {x_test.shape}, Rótulos de teste: {y_test.shape}")

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
### Carregando e exibindo informações sobre o MNIST ###
Dados de treino: (54000, 28, 28), Rótulos de treino: (54000,)
Dados de validação: (6000, 28, 28), Rótulos de validação: (6000,)
Dados de teste: (10000, 28, 28), Rótulos de teste: (10000,)


In [6]:
# Criação de tf.data.Dataset
train_dataset = tf.data.Dataset.from_tensor_slices((x_train, y_train)).shuffle(buffer_size=len(x_train)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_dataset = tf.data.Dataset.from_tensor_slices((x_val, y_val)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_dataset = tf.data.Dataset.from_tensor_slices((x_test, y_test)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

## **Etapa 2: Definição do Modelo**

1. **Função `create_mnist_model()`:** encapsula a criação de um modelo Keras Sequencial simples.
  * `Input` layer: define o formato da entrada (imagens 28x28).
  * `Flatten` layer: achata a imagem 2D para um vetor 1D.
  * `Dense` layers: camadas totalmente conectadas (uma oculta com ativação ReLU e uma de saída com 10 unidades para as classes, produzindo logits).
  * `model_custom_loop.summary()` para visualizar a arquitetura.

In [7]:
# Etapa 2. Definição do Modelo
def create_mnist_model():
    """Cria um modelo Sequencial simples para o MNIST."""
    model = Sequential([
        Input(shape=(28, 28), name="input_layer"),
        Flatten(name="flatten_layer"),
        Dense(128, activation="relu", name="dense_1"),
        Dense(10, name="output_layer") # Logits serão produzidos aqui
    ], name="mnist_model")
    return model

model_custom_loop = create_mnist_model()
model_custom_loop.summary()

Model: "mnist_model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten_layer (Flatten)         │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │       100,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output_layer (Dense)            │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 101,770 (397.54 KB)

 Trainable params: 101,770 (397.54 KB)

 Non-trainable params: 0 (0.00 B)

## **Etapa 3: Definição da Função de Perda, Otimizador e Métricas**

* **Função de Perda (`loss_fn`):** `SparseCategoricalCrossentropy(from_logits=True)` – adequada para classificação multiclasse com rótulos inteiros e saídas de logits.
* **Otimizador (`optimizer`):** `Adam(learning_rate=LEARNING_RATE)` – otimizador padrão.
* **Métricas:**
  * Para Treino: `Mean(name='train_loss')` para a perda média e `SparseCategoricalAccuracy(name='train_accuracy')` para a acurácia.
  * Para Validação: `Mean(name='val_loss')` e `SparseCategoricalAccuracy(name='val_accuracy')`.  
  

In [8]:
# Etapa 3. Definição da Função de Perda, Otimizador e Métricas
loss_fn = SparseCategoricalCrossentropy(from_logits=True)
optimizer = Adam(learning_rate=LEARNING_RATE)

# Métricas para o treino
train_loss_metric = Mean(name='train_loss')
train_accuracy_metric = SparseCategoricalAccuracy(name='train_accuracy')

# Métricas para a validação
val_loss_metric = Mean(name='val_loss')
val_accuracy_metric = SparseCategoricalAccuracy(name='val_accuracy')

## **Etapa 4: Treinamento** (`train_step`)

* **`@tf.function`** compila a função em um grafo TensorFlow para melhor desempenho.
* **Lógica do Passo de Treinamento:**
  1.  Uso de `tf.GradientTape()` para registrar as operações e calcular os gradientes.
  2.  Forward Pass: `logits = model_custom_loop(x_batch, training=True)`.
  3.  Cálculo da Perda: `loss_value = loss_fn(y_batch, logits)`.
  4.  Cálculo dos Gradientes: `grads = tape.gradient(loss_value, model_custom_loop.trainable_weights)`.
  5.  Aplicação dos Gradientes: `optimizer.apply_gradients(zip(grads, model_custom_loop.trainable_weights))`.
  6.  Atualização das Métricas de Treino: `train_loss_metric.update_state()` e `train_accuracy_metric.update_state()`.

In [9]:
# Etapa 4. Treinamento (Train Step) como uma `tf.function` para otimização**
@tf.function
def train_step(x_batch, y_batch):
    """Executa um passo de treinamento."""
    with tf.GradientTape() as tape:
        logits = model_custom_loop(x_batch, training=True)
        loss_value = loss_fn(y_batch, logits)

    # Adiciona outras perdas do modelo (ex: regularização), quando houver
    # loss_value += sum(model_custom_loop.losses) # Descomente se tiver camadas com perdas de regularização adicionadas via add_loss()

    grads = tape.gradient(loss_value, model_custom_loop.trainable_weights)
    optimizer.apply_gradients(zip(grads, model_custom_loop.trainable_weights))

    # Atualiza as métricas de treino
    train_loss_metric.update_state(loss_value)
    train_accuracy_metric.update_state(y_batch, logits)
    return loss_value

## **Etapa 5: Validação/Teste** (`test_step`)

  * Uso de **`@tf.function`** para otimização.
  * **Lógica do Passo de Avaliação:**
    1.  Forward Pass: `logits = model_custom_loop(x_batch, training=False)` (importante: `training=False` para desativar comportamentos específicos de treino como Dropout, se houver).
    2.  Cálculo da Perda: `loss_value = loss_fn(y_batch, logits)`.
    3.  Atualização das Métricas (de validação ou teste): `loss_metric.update_state()` e `acc_metric.update_state()`.

In [10]:
# Etapa 5. Passo de Validação/Teste (Evaluation Step) como uma `tf.function`
@tf.function
def test_step(x_batch, y_batch, loss_metric, acc_metric):
    """Executa um passo de avaliação (validação ou teste)."""
    logits = model_custom_loop(x_batch, training=False)
    loss_value = loss_fn(y_batch, logits)

    # Atualiza as métricas
    loss_metric.update_state(loss_value)
    acc_metric.update_state(y_batch, logits)

## **Etapa 6: Loop de Treinamento Personalizado**

* **Loop principal do treinamento** iteração por épocas.
* **Reset das Métricas:** `reset_state()` no início de cada época para as métricas de treino e validação.
* **Loop interno de treinamento** itera sobre `train_dataset`.
  * Chama `train_step()` para cada lote.
  * Log periódico do progresso (perda e acurácia do batch/acumulada).
* **Loop interno de validação** itera sobre `val_dataset`.
  * Chama `test_step()` para cada lote, usando as métricas de validação.

Ao final de cada época, imprime a perda e acurácia médias para treino e validação.

In [11]:
# Etapa 6. Loop de Treinamento Personalizado
print("\nIniciando o treinamento personalizado...")
for epoch in range(NUM_EPOCHS):
    print(f"--- Época {epoch + 1}/{NUM_EPOCHS} ---")

    # Resetar métricas no início de cada época
    train_loss_metric.reset_state()
    train_accuracy_metric.reset_state()
    val_loss_metric.reset_state()
    val_accuracy_metric.reset_state()

    # Loop de Treinamento
    for step, (x_batch_train, y_batch_train) in enumerate(train_dataset):
        loss_value = train_step(x_batch_train, y_batch_train)
        if step % 100 == 0: # Log a cada 100 batches
            print(f"  Batch {step}: Perda Treino (batch atual) = {loss_value:.4f}, Acurácia Treino (acumulada) = {train_accuracy_metric.result():.4f}")

    # Loop de Validação
    for x_batch_val, y_batch_val in val_dataset:
        test_step(x_batch_val, y_batch_val, val_loss_metric, val_accuracy_metric)

    # Exibir métricas ao final da época
    print(f"  Resultado da Época {epoch + 1}:")
    print(f"    Perda Treino: {train_loss_metric.result():.4f}, Acurácia Treino: {train_accuracy_metric.result():.4f}")
    print(f"    Perda Validação: {val_loss_metric.result():.4f}, Acurácia Validação: {val_accuracy_metric.result():.4f}")
    print("-" * 30)


Iniciando o treinamento personalizado...
--- Época 1/10 ---
  Batch 0: Perda Treino (batch atual) = 2.2614, Acurácia Treino (acumulada) = 0.1875
  Batch 100: Perda Treino (batch atual) = 0.2003, Acurácia Treino (acumulada) = 0.8032
  Batch 200: Perda Treino (batch atual) = 0.2886, Acurácia Treino (acumulada) = 0.8490
  Batch 300: Perda Treino (batch atual) = 0.2477, Acurácia Treino (acumulada) = 0.8701
  Batch 400: Perda Treino (batch atual) = 0.2124, Acurácia Treino (acumulada) = 0.8844
  Batch 500: Perda Treino (batch atual) = 0.1500, Acurácia Treino (acumulada) = 0.8938
  Batch 600: Perda Treino (batch atual) = 0.2348, Acurácia Treino (acumulada) = 0.9010
  Batch 700: Perda Treino (batch atual) = 0.1077, Acurácia Treino (acumulada) = 0.9068
  Batch 800: Perda Treino (batch atual) = 0.1255, Acurácia Treino (acumulada) = 0.9115
  Resultado da Época 1:
    Perda Treino: 0.3100, Acurácia Treino: 0.9131
    Perda Validação: 0.1690, Acurácia Validação: 0.9530
----------------------------

In [12]:
# @title **Etapa 7. Avaliação Final no Conjunto de Teste**
print("\nAvaliando o modelo no conjunto de teste...")

# Avalia o desempenho do modelo treinado em dados completamente não vistos
test_loss_metric = Mean(name='test_loss')
test_accuracy_metric = SparseCategoricalAccuracy(name='test_accuracy')

for x_batch_test, y_batch_test in test_dataset:
    test_step(x_batch_test, y_batch_test, test_loss_metric, test_accuracy_metric)

# Imprime a perda e acurácia finais no conjunto de teste
print(f"Resultado Final no Teste:")
print(f"  Perda Teste: {test_loss_metric.result():.4f}")
print(f"  Acurácia Teste: {test_accuracy_metric.result():.4f}")


Avaliando o modelo no conjunto de teste...
Resultado Final no Teste:
  Perda Teste: 0.0741
  Acurácia Teste: 0.9765


## **Parte 2: Ajuste de Hiperparâmetros com KerasTuner** (usando `model.fit`)

### **Etapa 1: Preparação de dados sintéticos para classificação binária**

* **Geração de dados** com o `make_classification` do Scikit-learn para criar um dataset de exemplo para classificação binária.
* **Divisão:** `train_test_split` para criar conjuntos de treino (`X_train_synth`, `y_train_synth`) e validação (`X_val_synth`, `y_val_synth`).
Esta parte demonstra o uso mais comum do KerasTuner com `model.fit()`.

In [13]:
# @title **Etapa 1. Preparação de Dados (Exemplo com dataset sintético para classificação binária)**
X_synth, y_synth = make_classification(n_samples=1000, n_features=20, n_classes=2, random_state=RANDOM_SEED)
X_train_synth, X_val_synth, y_train_synth, y_val_synth = train_test_split(
    X_synth, y_synth, test_size=0.2, random_state=RANDOM_SEED
)

print("\n### Iniciando Ajuste de Hiperparâmetros com KerasTuner (model.fit) ###")
print(f"Dados de treino: {x_train.shape}, Rótulos de treino: {y_train.shape}")
print(f"Dados de validação: {x_val.shape}, Rótulos de validação: {y_val.shape}")
print(f"Dados de teste: {x_test.shape}, Rótulos de teste: {y_test.shape}")


### Iniciando Ajuste de Hiperparâmetros com KerasTuner (model.fit) ###
Dados de treino: (54000, 28, 28), Rótulos de treino: (54000,)
Dados de validação: (6000, 28, 28), Rótulos de validação: (6000,)
Dados de teste: (10000, 28, 28), Rótulos de teste: (10000,)


## **Etapa 2: Função de Construção do Modelo de Classificação**

* **Parâmetro `hp`:** A função recebe um objeto `HyperParameters` do KerasTuner.
* **Definição de hiperparâmetros:**
  * `hp.Int('units', ...)`: número de neurônios na primeira camada densa.
  * `hp.Float('learning_rate', ...)`: taxa de aprendizado para o otimizador Adam.
  * `hp.Boolean("add_another_layer")`: controla condicionalmente a adição de uma segunda camada densa.
  * `hp.Int('units_extra_layer', ...)`: número de neurônios para a camada extra (se adicionada).
  
* **Construção do modelo sequencial Keras**
  * Camada de entrada explícita.
  * Camadas densas com base nos hiperparâmetros.
  * Camada de saída sigmoide para classificação binária.

* **Compilação do Modelo:** `model.compile()` com o otimizador Adam (usando `hp_learning_rate`), perda `binary_crossentropy` e métrica `accuracy`.

In [14]:
# Etapa 2. Arquitetura do Modelo para o KerasTuner**
def build_classification_model(hp):
    """Constrói um modelo de classificação para o KerasTuner."""

    # Hiperparâmetros:
    # Número de unidades na primeira camada densa
    hp_units = hp.Int('units', min_value=32, max_value=256, step=32)
    # Taxa de aprendizado
    hp_learning_rate = hp.Float('learning_rate', min_value=1e-4, max_value=1e-2, sampling='log')

    model = Sequential(name=f"tuned_model_units_{hp.get('units')}_lr_{hp.get('learning_rate'):.0e}")

    # Camada de entrada (input_shape será inferido ou pode ser especificado)
    model.add(Input(shape=(X_train_synth.shape[1],)))
    model.add(Dense(units=hp_units, activation='relu'))

    # Hiperparâmetro: adicionar ou não uma segunda camada densa
    if hp.Boolean("add_another_layer"):
        model.add(Dense(units=hp.Int('units_extra_layer', min_value=16, max_value=128, step=16), activation='relu'))

    # Camada de saída
    model.add(Dense(1, activation='sigmoid')) # Classificação binária

    # Compila o modelo
    model.compile(optimizer=Adam(learning_rate=hp_learning_rate),
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    return model

In [15]:
# @title **Etapa 3: Configuração do Tuner**
# Aplicação do Hyperband, algoritmo mais avançado que o RandomSearch
tuner = kt.Hyperband(
    build_classification_model,
    objective='val_accuracy',
    max_epochs=20, # Máximo de épocas para treinar um modelo individualmente
    factor=3, # Fator de redução para o número de modelos e épocas por rodada
    hyperband_iterations=1, # Número de vezes que o algoritmo Hyperband será executado
    directory='keras_tuner_results',
    project_name='classification_tuning',
    seed=RANDOM_SEED
)

# Callback para parada antecipada dentro da busca do tuner
stop_early = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5)

print("\nIniciando a busca por hiperparâmetros com KerasTuner...")
tuner.search_space_summary() # Exibe o espaço de busca configurado

tuner.search(X_train_synth, y_train_synth,
             epochs=50, # Este 'epochs' é o número máximo de épocas que o Hyperband pode usar internamente por modelo, mas ele gerencia isso.
             validation_data=(X_val_synth, y_val_synth),
             callbacks=[stop_early])

Trial 30 Complete [00h 00m 08s]
val_accuracy: 0.8650000095367432

Best val_accuracy So Far: 0.8700000047683716
Total elapsed time: 00h 01m 44s


In [16]:
# @title **Etapa 4. Obteção dos melhores hiperparâmetros para o modelo**
print("\nBusca concluída. Obtendo os melhores resultados...")
tuner.results_summary(num_trials=5) # Exibe um resumo dos 5 melhores trials

best_hps_list = tuner.get_best_hyperparameters(num_trials=1)
if best_hps_list:
    best_hps = best_hps_list[0]
    print(f"\nMelhores Hiperparâmetros encontrados:")
    for hp_name, hp_value in best_hps.values.items():
        print(f"  {hp_name}: {hp_value}")

    # Construir o melhor modelo com os hiperparâmetros encontrados
    best_model = tuner.hypermodel.build(best_hps)
    best_model.summary()

    # Treinar o melhor modelo por mais épocas ou com dados completos
    # Exemplo: treinar por mais algumas épocas
    print("\nTreinando o melhor modelo encontrado...")
    history = best_model.fit(X_train_synth, y_train_synth,
                             epochs=50, # Pode ser um número maior de épocas aqui
                             validation_data=(X_val_synth, y_val_synth),
                             callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, verbose=1)]) # Novo EarlyStopping para o treino final

    # Avaliar o melhor modelo no conjunto de teste sintético (se houver)
    # val_loss, val_accuracy = best_model.evaluate(X_val_synth, y_val_synth)
    # print(f"\nAvaliação do melhor modelo no conjunto de validação: Perda={val_loss:.4f}, Acurácia={val_accuracy:.4f}")
else:
    print("Nenhum hiperparâmetro foi encontrado. Verifique as configurações do tuner.")


Busca concluída. Obtendo os melhores resultados...
Results summary
Results in keras_tuner_results/classification_tuning
Showing 5 best trials
Objective(name="val_accuracy", direction="max")

Trial 0014 summary
Hyperparameters:
units: 256
learning_rate: 0.0007767449713530701
add_another_layer: False
units_extra_layer: 64
tuner/epochs: 7
tuner/initial_epoch: 3
tuner/bracket: 2
tuner/round: 1
tuner/trial_id: 0006
Score: 0.8700000047683716

Trial 0020 summary
Hyperparameters:
units: 160
learning_rate: 0.0033137745378583987
add_another_layer: False
units_extra_layer: 32
tuner/epochs: 7
tuner/initial_epoch: 0
tuner/bracket: 1
tuner/round: 0
Score: 0.8700000047683716

Trial 0024 summary
Hyperparameters:
units: 160
learning_rate: 0.0033137745378583987
add_another_layer: False
units_extra_layer: 32
tuner/epochs: 20
tuner/initial_epoch: 7
tuner/bracket: 1
tuner/round: 1
tuner/trial_id: 0020
Score: 0.8700000047683716

Trial 0022 summary
Hyperparameters:
units: 64
learning_rate: 0.009080300897480

Model: "tuned_model_units_256_lr_8e-04"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                 │ (None, 256)            │         5,376 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1)              │           257 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,633 (22.00 KB)

 Trainable params: 5,633 (22.00 KB)

 Non-trainable params: 0 (0.00 B)


Treinando o melhor modelo encontrado...
Epoch 1/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.6830 - loss: 0.6278 - val_accuracy: 0.7950 - val_loss: 0.5411
Epoch 2/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8782 - loss: 0.4491 - val_accuracy: 0.8450 - val_loss: 0.4442
Epoch 3/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8927 - loss: 0.3753 - val_accuracy: 0.8450 - val_loss: 0.3983
Epoch 4/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8852 - loss: 0.3384 - val_accuracy: 0.8600 - val_loss: 0.3753
Epoch 5/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8867 - loss: 0.3184 - val_accuracy: 0.8650 - val_loss: 0.3634
Epoch 6/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8847 - loss: 0.3063 - val_accuracy: 0.8600 - val_loss: 0.3572
Epoch 7/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8880 - loss: 0.2979 - val_accuracy: 0.8650 - val_loss: 0.3540
Epoch 8/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8867 - loss:

# **Referências**

* ABADI,  Martín _et al_. **TensorFlow: Large-Scale Machine Learning on Heterogeneous Distributed Systems**. Preliminary White Paper, 9 nov. 2015. arXiv:1603.04467v2 [cs.DC], 16 mar. 2016. Disponível em: https://arxiv.org/pdf/1603.04467 Acesso em: 05 Maio. 2025.

* PASZKE, A. et al. **PyTorch: An Imperative Style, High-Performance Deep Learning Library**. arXiv preprint arXiv:1912.01703 , 2019. Disponível em: https://arxiv.org/abs/1912.01703 .Acesso em: 06 Maio. 2025.

* SNOEK, J.; LAROCHELLE, H.; ADAMS, R. P. **Practical Bayesian Optimization of Machine Learning Algorithms**. Advances in Neural Information Processing Systems 25, 2012. Disponível em: https://proceedings.neurips.cc/paper/2012/file/05311655a15b75fab86956663e1819cd-Paper.pdf
Acesso em: 02 Maio. 2025.

* LI, L. et al. **Hyperband: Bandit-based configuration evaluation for hyperparameter optimization**. In: International Conference on Learning Representations (ICLR) , 2017. Disponível em:

* LI, L. et al. **Hyperband: A Novel Bandit-Based Approach to Hyperparameter Optimization**. Journal of Machine Learning Research, v. 18, p. 1-52, 2018. Disponível em: https://www.jmlr.org/papers/volume18/16-558/16-558.pdf Acesso em: 11 Maio. 2025.



* Zoph, B., et al. (2018). *Learning Transferable Architectures for Scalable Image Recognition*.